### Import Packages

In [1]:
import duckdb
from pathlib import Path

#### Configs and Variables

In [2]:
data_vault_path = r"C:\Users\JacksonDivakarRajasi\OneDrive - JMAN Group Ltd\Documents\Python Folder\DuckDB Environment\Olist Eccomerce DuckDB\DuckDB Data\data_vault.duckdb"

#### DuckDB Connection

In [3]:
con = duckdb.connect(data_vault_path)

In [4]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS mart;
""")

### Fact Order Item

In [5]:
con.execute("""
CREATE OR REPLACE TABLE mart.fact_order_item AS
SELECT
    b.order_item_hk,
    b.order_hk,
    b.order_item_id,

    -- Latest trusted dimension versions
    dc.customer_sk,
    dp.product_sk,
    ds.seller_sk,

    -- Date Key
    CAST(
        STRFTIME(o.order_purchase_timestamp, '%Y%m%d')
        AS INTEGER
    ) AS order_date_key,

    -- Dates
    o.order_purchase_timestamp,
    oi.shipping_limit_date,

    -- Measures
    oi.price,
    oi.freight_value,

    COALESCE(oi.price, 0)
        + COALESCE(oi.freight_value, 0)
        AS total_item_value

FROM business_vault.bridge_order_item b

JOIN business_vault.current_order_item oi
    ON b.order_item_hk = oi.order_item_hk

JOIN business_vault.order_lifecycle_current o
    ON b.order_hk = o.order_hk

LEFT JOIN mart.dim_customer dc
    ON b.customer_hk = dc.customer_hk
   AND dc.is_current = TRUE

LEFT JOIN mart.dim_product dp
    ON b.product_hk = dp.product_hk
   AND dp.is_current = TRUE

LEFT JOIN mart.dim_seller ds
    ON b.seller_hk = ds.seller_hk
   AND ds.is_current = TRUE;
""")

### Fact Payment

In [6]:
con.execute("""
CREATE OR REPLACE TABLE mart.fact_payment AS
SELECT
    b.order_payment_hk,
    b.order_hk,
    b.payment_sequential,

    -- Customer Dimension
    dc.customer_sk,

    -- Date
    CAST(
        STRFTIME(o.order_purchase_timestamp, '%Y%m%d')
        AS INTEGER
    ) AS order_date_key,

    o.order_purchase_timestamp,

    -- Payment attributes / measures
    p.payment_type,
    p.payment_installments,
    p.payment_value

FROM business_vault.bridge_order_payment b

JOIN business_vault.current_order_payment p
    ON b.order_payment_hk = p.order_payment_hk

JOIN business_vault.order_lifecycle_current o
    ON b.order_hk = o.order_hk

LEFT JOIN mart.dim_customer dc
    ON b.customer_hk = dc.customer_hk
   AND dc.is_current = TRUE;
""")

### Fact Order Lifecycle

In [7]:
con.execute("""
CREATE OR REPLACE TABLE mart.fact_order_lifecycle AS
SELECT
    b.order_hk,

    -- Customer Dimension
    dc.customer_sk,

    -- Purchase Date
    CAST(
        STRFTIME(o.order_purchase_timestamp, '%Y%m%d')
        AS INTEGER
    ) AS purchase_date_key,

    -- Approval Date
    CASE
        WHEN o.order_approved_at IS NOT NULL
        THEN CAST(
            STRFTIME(o.order_approved_at, '%Y%m%d')
            AS INTEGER
        )
    END AS approved_date_key,

    -- Carrier Date
    CASE
        WHEN o.order_delivered_carrier_date IS NOT NULL
        THEN CAST(
            STRFTIME(
                o.order_delivered_carrier_date,
                '%Y%m%d'
            ) AS INTEGER
        )
    END AS carrier_date_key,

    -- Actual Delivery Date
    CASE
        WHEN o.order_delivered_customer_date IS NOT NULL
        THEN CAST(
            STRFTIME(
                o.order_delivered_customer_date,
                '%Y%m%d'
            ) AS INTEGER
        )
    END AS delivery_date_key,

    -- Estimated Delivery Date
    CASE
        WHEN o.order_estimated_delivery_date IS NOT NULL
        THEN CAST(
            STRFTIME(
                o.order_estimated_delivery_date,
                '%Y%m%d'
            ) AS INTEGER
        )
    END AS estimated_delivery_date_key,

    -- Status
    o.order_status,

    -- Lifecycle measures
    o.hours_to_approve,
    o.hours_to_carrier,
    o.days_to_deliver,
    o.delivery_delay_days,

    -- Business classification
    o.delivery_status

FROM business_vault.bridge_order b

JOIN business_vault.order_lifecycle_current o
    ON b.order_hk = o.order_hk

LEFT JOIN mart.dim_customer dc
    ON b.customer_hk = dc.customer_hk
   AND dc.is_current = TRUE;
""")